In [5]:
!pip install pynput

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import cv2
import cvzone
from cvzone.HandTrackingModule import HandDetector
from pynput.keyboard import Controller

keyboard = Controller()

detector = HandDetector(detectionCon=0.8)

cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)


class Button:
    def __init__(self, pos, text, size=[85, 85]):
        self.pos = pos
        self.size = size
        self.text = text

keys = [
    ["Q","W","E","R","T","Y","U","I","O","P"],
    ["A","S","D","F","G","H","J","K","L",";"],
    ["Z","X","C","V","B","N","M",",",".","/"]
]

buttonList = []

for i in range(len(keys)):
    for j, key in enumerate(keys[i]):
        buttonList.append(Button([100 * j + 50, 100 * i + 50], key))

def drawAll(img, buttonList):
    for button in buttonList:
        x, y = button.pos
        w, h = button.size

        cvzone.cornerRect(
            img,
            (x, y, w, h),
            20,
            rt=0
        )

        cv2.rectangle(
            img,
            (x, y),
            (x + w, y + h),
            (255,0,0),
            cv2.FILLED
        )

        cv2.putText(
            img,
            button.text,
            (x + 20, y + 65),
            cv2.FONT_HERSHEY_PLAIN,
            4,
            (255, 255, 255),
            4
        )

    return img

clicked = False

while True:
    success, img = cap.read()

    if not success:
        break
    img = cv2.flip(img, 1)

    hands, img = detector.findHands(img)

    img = drawAll(img, buttonList)

    if hands:

        hand = hands[0]
        lmList = hand["lmList"]

        for button in buttonList:

            x, y = button.pos
            w, h = button.size

            # Hover effect
            if x < lmList[8][0] < x + w and y < lmList[8][1] < y + h:

                cv2.rectangle(
                    img,
                    (x - 5, y - 5),
                    (x + w + 5, y + h + 5),
                    (175, 0, 175),
                    cv2.FILLED
                )

                cv2.putText(
                    img,
                    button.text,
                    (x + 20, y + 65),
                    cv2.FONT_HERSHEY_PLAIN,
                    4,
                    (255, 255, 255),
                    4
                )

                # Distance between index tip and middle tip
                l, info, img = detector.findDistance(
                    lmList[4][:2],
                    lmList[8][:2],
                    img
                )

                # Click only once when fingers come together
                if l < 50 and not clicked:

                    keyboard.press(button.text)
                    keyboard.release(button.text)

                    clicked = True

                    cv2.rectangle(
                        img,
                        (x, y),
                        (x + w, y + h),
                        (0, 0, 255),
                        cv2.FILLED
                    )

                    cv2.putText(
                        img,
                        button.text,
                        (x + 20, y + 65),
                        cv2.FONT_HERSHEY_PLAIN,
                        4,
                        (255, 255, 255),
                        4
                    )

                    print(f"Pressed: {button.text}")

        # Reset click only after fingers separate
        l, _, _ = detector.findDistance(
            lmList[4][:2],
            lmList[8][:2],
            img
        )

        if l > 60:
            clicked = False

    cv2.imshow("Virtual Keyboard", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Pressed: C
Pressed: C
Pressed: C
Pressed: C
Pressed: C
Pressed: C
Pressed: F
